# 01 Data Cleaning

This notebook cleans and merges seven datasets into a single country-year panel for analysis:

1. **ACLED** — conflict events and fatalities (Africa only)
2. **World Bank** — employment in agriculture, industry, services
3. **World Bank** — GDP per capita
4. **World Bank** — population density
5. **V-Dem** — rule of law index

All datasets are filtered to **African countries only** before merging, using the country list derived from the ACLED source data.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Preferred data locations (local first, then mounted drive).
preferred_data_paths = [
    Path("/Users/xuyingnuo/Desktop/project_folder/data"),
    Path("/content/drive/MyDrive/project_folder/data"),
]

# User-overridable path.
path_to_data_folder = str(globals().get("path_to_data_folder", "")).strip()
if not path_to_data_folder:
    for p in preferred_data_paths:
        if p.exists():
            path_to_data_folder = str(p)
            break


def find_project_root(data_dir: Path) -> Path:
    """Find project root given a data directory (expects data/acled_data.csv)."""
    candidate_roots = []

    if data_dir.exists():
        candidate_roots.append(data_dir.parent)

    candidate_roots.extend(
        [
            Path(str(globals().get("project_folder", ""))),
            Path(str(globals().get("resolved_root", ""))),
            Path("/Users/xuyingnuo/Desktop/project_folder"),
            Path("/content/drive/MyDrive/project_folder"),
            Path("/workspaces/project_folder"),
            Path("/workspace/project_folder"),
            Path.cwd(),
            Path.cwd().parent,
        ]
    )

    for root in candidate_roots:
        if str(root) in {"", "."}:
            continue
        if (root / "data" / "acled_data.csv").exists():
            return root

    # Fallback: infer root from a discovered acled_data.csv anywhere accessible.
    search_bases = [Path.cwd(), Path.home(), Path("/Users"), Path("/content")]
    for base in search_bases:
        if not base.exists():
            continue
        try:
            match = next(base.rglob("acled_data.csv"), None)
            if match is not None:
                return match.parent.parent
        except Exception:
            continue

    return Path.cwd()


# Build DATA_DIR from explicit path if valid, otherwise from discovered ROOT.
explicit_data_dir = Path(path_to_data_folder).expanduser() if path_to_data_folder else Path("")
if explicit_data_dir.exists() and (explicit_data_dir / "acled_data.csv").exists():
    DATA_DIR = explicit_data_dir.resolve()
    ROOT = DATA_DIR.parent
else:
    ROOT = find_project_root(explicit_data_dir if explicit_data_dir else Path("."))
    DATA_DIR = ROOT / "data"

OUT_DIR = ROOT / "outputs"
CLEAN_DIR = OUT_DIR / "cleaned"

OUT_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print(f"Root: {ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"acled_data.csv exists: {(DATA_DIR / 'acled_data.csv').exists()}")
print(f"Output dir: {OUT_DIR}")

if not (DATA_DIR / "acled_data.csv").exists():
    raise FileNotFoundError(
        "Could not find acled_data.csv. "
        "Use a local VS Code kernel or mount Google Drive and set path_to_data_folder "
        "to /content/drive/MyDrive/project_folder/data."
    )

Root: /Users/xuyingnuo/Desktop/project_folder
Data dir: /Users/xuyingnuo/Desktop/project_folder/data
acled_data.csv exists: True
Output dir: /Users/xuyingnuo/Desktop/project_folder/outputs


In [2]:
# Quick 20-second environment check before running all cells
import os
import sys

print("1) CWD:", os.getcwd())
print("2) Python:", sys.executable)

local_data = Path("/Users/xuyingnuo/Desktop/project_folder/data")
drive_data = Path("/content/drive/MyDrive/project_folder/data")
print("3) Local data path exists:", local_data.exists())
print("4) Drive data path exists:", drive_data.exists())

print("5) DATA_DIR:", DATA_DIR)
print("6) acled_data.csv exists in DATA_DIR:", (DATA_DIR / "acled_data.csv").exists())

if (DATA_DIR / "acled_data.csv").exists():
    sample = pd.read_csv(DATA_DIR / "acled_data.csv", nrows=3)
    print("7) ACLED sample shape:", sample.shape)
    print("8) ACLED columns:", list(sample.columns))
else:
    print("7) ACLED sample check skipped: file not found in DATA_DIR")

1) CWD: /Users/xuyingnuo/Desktop/project_folder/code
2) Python: /Users/xuyingnuo/Desktop/project_folder/.venv/bin/python
3) Local data path exists: True
4) Drive data path exists: False
5) DATA_DIR: /Users/xuyingnuo/Desktop/project_folder/data
6) acled_data.csv exists in DATA_DIR: True
7) ACLED sample shape: (3, 13)
8) ACLED columns: ['WEEK', 'REGION', 'COUNTRY', 'ADMIN1', 'EVENT_TYPE', 'SUB_EVENT_TYPE', 'EVENTS', 'FATALITIES', 'POPULATION_EXPOSURE', 'DISORDER_TYPE', 'ID', 'CENTROID_LATITUDE', 'CENTROID_LONGITUDE']


In [3]:
def to_snake(name: str) -> str:
    return (
        str(name)
        .strip()
        .lower()
        .replace("%", "pct")
        .replace("(", "")
        .replace(")", "")
        .replace("-", "_")
        .replace("/", "_")
        .replace(" ", "_")
    )


def clean_wdi_file(path: Path, value_name: str, start_year: int = 1996, end_year: int = 2023) -> pd.DataFrame:
    # World Bank files include 4 metadata lines before the actual header row.
    df = pd.read_csv(path, skiprows=4)
    df.columns = [c.strip() for c in df.columns]

    id_cols = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]
    year_cols = [c for c in df.columns if c.isdigit() and start_year <= int(c) <= end_year]

    keep_cols = [c for c in id_cols if c in df.columns] + year_cols
    df = df[keep_cols].copy()

    long_df = df.melt(
        id_vars=[c for c in ["Country Name", "Country Code"] if c in df.columns],
        value_vars=year_cols,
        var_name="year",
        value_name=value_name,
    )

    long_df.columns = [to_snake(c) for c in long_df.columns]
    long_df["year"] = pd.to_numeric(long_df["year"], errors="coerce").astype("Int64")
    long_df[value_name] = pd.to_numeric(long_df[value_name], errors="coerce")

    for c in ["country_name", "country_code"]:
        if c in long_df.columns:
            long_df[c] = long_df[c].astype(str).str.strip()

    long_df = long_df.dropna(subset=[value_name, "year"]).drop_duplicates()
    return long_df


# 1) ACLED
acled_raw = pd.read_csv(DATA_DIR / "acled_data.csv")
acled = acled_raw.copy()
acled.columns = [to_snake(c) for c in acled.columns]

if "week" in acled.columns:
    acled["week"] = pd.to_datetime(acled["week"], format="%d-%B-%Y", errors="coerce")
    acled["year"] = acled["week"].dt.year.astype("Int64")

for c in ["events", "fatalities", "population_exposure", "centroid_latitude", "centroid_longitude"]:
    if c in acled.columns:
        acled[c] = pd.to_numeric(acled[c], errors="coerce")

for c in acled.select_dtypes(include="object").columns:
    acled[c] = acled[c].str.strip()

acled = acled.drop_duplicates()

# ── Derive the list of African countries from ACLED ──
# ACLED only covers African countries, so its unique country names define our scope.
african_countries = set(acled["country"].dropna().unique())
print(f"African countries from ACLED: {len(african_countries)}")


# 2-6) World Bank indicator files
agri = clean_wdi_file(DATA_DIR / "employment_in_agriculture.csv", "employment_agriculture_pct")
ind = clean_wdi_file(DATA_DIR / "employment_in_industry.csv", "employment_industry_pct")
serv = clean_wdi_file(DATA_DIR / "employment_in_services.csv", "employment_services_pct")
gdp = clean_wdi_file(DATA_DIR / "gdp_per_capita.csv", "gdp_per_capita_constant_2015_usd")
pop = clean_wdi_file(DATA_DIR / "population_density.csv", "population_density_per_sq_km")


# 7) V-Dem rule index
vdem_raw = pd.read_csv(DATA_DIR / "vdem_1996-2023_rule.csv", low_memory=False)
vdem_raw.columns = [to_snake(c) for c in vdem_raw.columns]

country_candidates = ["country_name", "country_text_id", "country"]
year_candidates = ["year"]
rule_candidates = ["v2x_rule", "rule", "rule_of_law"]

country_col = next((c for c in country_candidates if c in vdem_raw.columns), None)
year_col = next((c for c in year_candidates if c in vdem_raw.columns), None)
rule_col = next((c for c in rule_candidates if c in vdem_raw.columns), None)

if country_col is None or year_col is None or rule_col is None:
    raise ValueError(
        f"Could not identify required V-Dem columns. Found country={country_col}, year={year_col}, rule={rule_col}."
    )

vdem = vdem_raw[[country_col, year_col, rule_col]].copy()
vdem = vdem.rename(
    columns={
        country_col: "country_name",
        year_col: "year",
        rule_col: "vdem_rule_index",
    }
)

vdem["country_name"] = vdem["country_name"].astype(str).str.strip()
vdem["year"] = pd.to_numeric(vdem["year"], errors="coerce").astype("Int64")
vdem["vdem_rule_index"] = pd.to_numeric(vdem["vdem_rule_index"], errors="coerce")
vdem = vdem[(vdem["year"] >= 1996) & (vdem["year"] <= 2023)]
vdem = vdem.dropna(subset=["country_name", "year", "vdem_rule_index"]).drop_duplicates()


# ── Filter all datasets to African countries only ──
def filter_africa(df_wb: pd.DataFrame, african_set: set) -> pd.DataFrame:
    """Keep only rows whose country_name appears in the African country set."""
    before = len(df_wb)
    filtered = df_wb[df_wb["country_name"].isin(african_set)].copy()
    after = len(filtered)
    print(f"  Africa filter: {before:,} → {after:,} rows (removed {before - after:,})")
    return filtered

print("\nFiltering World Bank and V-Dem datasets to African countries:")
agri = filter_africa(agri, african_countries)
ind = filter_africa(ind, african_countries)
serv = filter_africa(serv, african_countries)
gdp = filter_africa(gdp, african_countries)
pop = filter_africa(pop, african_countries)
vdem = filter_africa(vdem, african_countries)

print("\nCleaning complete (Africa only).")
print(f"ACLED rows: {len(acled):,}")
print(f"Agriculture rows: {len(agri):,}")
print(f"Industry rows: {len(ind):,}")
print(f"Services rows: {len(serv):,}")
print(f"GDP rows: {len(gdp):,}")
print(f"Population rows: {len(pop):,}")
print(f"V-Dem rows: {len(vdem):,}")

/var/folders/vk/zyg9c8p51p14tm8r139z9gcr0000gn/T/ipykernel_9524/3704250055.py:58: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in acled.select_dtypes(include="object").columns:


African countries from ACLED: 58

Filtering World Bank and V-Dem datasets to African countries:
  Africa filter: 6,576 → 1,259 rows (removed 5,317)
  Africa filter: 6,576 → 1,259 rows (removed 5,317)
  Africa filter: 6,576 → 1,259 rows (removed 5,317)
  Africa filter: 7,089 → 1,239 rows (removed 5,850)
  Africa filter: 7,237 → 1,256 rows (removed 5,981)
  Africa filter: 4,953 → 1,385 rows (removed 3,568)

Cleaning complete (Africa only).
ACLED rows: 268,511
Agriculture rows: 1,259
Industry rows: 1,259
Services rows: 1,259
GDP rows: 1,239
Population rows: 1,256
V-Dem rows: 1,385


In [4]:
# Save cleaned individual datasets
acled.to_csv(CLEAN_DIR / "acled_data_clean.csv", index=False)
agri.to_csv(CLEAN_DIR / "employment_in_agriculture_clean.csv", index=False)
ind.to_csv(CLEAN_DIR / "employment_in_industry_clean.csv", index=False)
serv.to_csv(CLEAN_DIR / "employment_in_services_clean.csv", index=False)
gdp.to_csv(CLEAN_DIR / "gdp_per_capita_clean.csv", index=False)
pop.to_csv(CLEAN_DIR / "population_density_clean.csv", index=False)
vdem.to_csv(CLEAN_DIR / "vdem_rule_clean.csv", index=False)

# Build a country-year panel from socio-economic + governance indicators
panel = (
    agri.merge(ind, on=["country_name", "country_code", "year"], how="outer")
    .merge(serv, on=["country_name", "country_code", "year"], how="outer")
    .merge(gdp, on=["country_name", "country_code", "year"], how="outer")
    .merge(pop, on=["country_name", "country_code", "year"], how="outer")
)

# ACLED aggregated to country-year level
acled_country_year = (
    acled.dropna(subset=["country", "year"])
    .groupby(["country", "year"], as_index=False)
    .agg(
        acled_events=("events", "sum"),
        acled_fatalities=("fatalities", "sum"),
        acled_population_exposure_mean=("population_exposure", "mean"),
    )
    .rename(columns={"country": "country_name"})
)

cleaned_enhanced = (
    panel.merge(vdem, on=["country_name", "year"], how="left")
    .merge(acled_country_year, on=["country_name", "year"], how="left")
    .sort_values(["country_name", "year"]) 
    .reset_index(drop=True)
)

# Final safety filter: keep only African countries
cleaned_enhanced = cleaned_enhanced[cleaned_enhanced["country_name"].isin(african_countries)].reset_index(drop=True)

cleaned_enhanced.to_csv(OUT_DIR / "cleaned_enhanced_data.csv", index=False)

print(f"Saved cleaned datasets to: {CLEAN_DIR}")
print(f"Saved merged dataset: {OUT_DIR / 'cleaned_enhanced_data.csv'}")
print(f"Merged rows (Africa only): {len(cleaned_enhanced):,}")
print(f"Unique African countries: {cleaned_enhanced['country_name'].nunique()}")

Saved cleaned datasets to: /Users/xuyingnuo/Desktop/project_folder/outputs/cleaned
Saved merged dataset: /Users/xuyingnuo/Desktop/project_folder/outputs/cleaned_enhanced_data.csv
Merged rows (Africa only): 1,288
Unique African countries: 46


In [5]:
report = {
    "cleaned_files": {
        "acled_data_clean.csv": int(len(acled)),
        "employment_in_agriculture_clean.csv": int(len(agri)),
        "employment_in_industry_clean.csv": int(len(ind)),
        "employment_in_services_clean.csv": int(len(serv)),
        "gdp_per_capita_clean.csv": int(len(gdp)),
        "population_density_clean.csv": int(len(pop)),
        "vdem_rule_clean.csv": int(len(vdem)),
        "cleaned_enhanced_data.csv": int(len(cleaned_enhanced)),
    },
    "year_range": {
        "min_year": int(cleaned_enhanced["year"].min()) if not cleaned_enhanced.empty else None,
        "max_year": int(cleaned_enhanced["year"].max()) if not cleaned_enhanced.empty else None,
    },
    "created_at": pd.Timestamp.utcnow().isoformat(),
}

with open(OUT_DIR / "summary_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

with open(OUT_DIR / "summary_report.txt", "w", encoding="utf-8") as f:
    f.write("Data cleaning summary\n")
    f.write("=" * 60 + "\n")
    for name, rows in report["cleaned_files"].items():
        f.write(f"{name}: {rows:,} rows\n")
    f.write("\n")
    f.write(f"Year range: {report['year_range']['min_year']} - {report['year_range']['max_year']}\n")

report

/var/folders/vk/zyg9c8p51p14tm8r139z9gcr0000gn/T/ipykernel_9524/811607324.py:16: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "created_at": pd.Timestamp.utcnow().isoformat(),


{'cleaned_files': {'acled_data_clean.csv': 268511,
  'employment_in_agriculture_clean.csv': 1259,
  'employment_in_industry_clean.csv': 1259,
  'employment_in_services_clean.csv': 1259,
  'gdp_per_capita_clean.csv': 1239,
  'population_density_clean.csv': 1256,
  'vdem_rule_clean.csv': 1385,
  'cleaned_enhanced_data.csv': 1288},
 'year_range': {'min_year': 1996, 'max_year': 2023},
 'created_at': '2026-04-24T07:59:16.995691+00:00'}